In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import math
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

Device configuration

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Model setup

In [ ]:
# Shannon entropy calculation
def shannon_entropy(text):
    if len(text) == 0:
        return 0.0

    # Count the frequency of each character in the text
    char_counts = Counter(text)
    total_chars = len(text)

    # Calculate the probability of each character
    probabilities = [count / total_chars for count in char_counts.values()]

    # Calculate Shannon entropy using the formula: H = -sum(p * log2(p))
    entropy = -sum(p * math.log2(p) for p in probabilities)

    return entropy

In [ ]:
# KL divergence calculation
def kl_divergence(p, q):
    p = np.array(p, dtype=np.float64) + 1e-10  # Add a small value to avoid division by zero
    q = np.array(q, dtype=np.float64) + 1e-10
    return np.sum(np.where(p != 0, p * np.log2(p / q), 0))

In [ ]:
# Feature extraction using CountVectorizer
class EntropyKLFeatureExtractor:
    def __init__(self, benign_texts):
        self.vectorizer = CountVectorizer()
        self.vectorizer.fit(benign_texts)

        benign_counts = self.vectorizer.transform(benign_texts).toarray()
        self.benign_distribution = np.mean(benign_counts, axis=0) / (np.sum(benign_counts) + 1e-10)  # Add a small value to avoid division by zero

    def extract_features(self, text):
        global_entropy = shannon_entropy(' '.join(text))

        vec = self.vectorizer.transform(text).toarray()[0]
        dist = vec / (np.sum(vec) + 1e-10)
        kl_div = kl_divergence(dist, self.benign_distribution)

        specical_ratio = sum(not c.isalnum() for c in text) / (len(text) + 1e-10)

        return np.array([
            global_entropy,
            kl_div,
            specical_ratio,
            len(text)],
            dtype=np.float32)

Model Training

In [ ]:
# Neural network model
class EntropyClassifier(nn.Module):
    def __init__(self, input_dim):
        super(EntropyClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [ ]:
# Dataset loading and preprocessing
file_path = 'dataset.csv'  # Path to dataset file
def load_dataset(file_path):
    df = pd.read_csv(file_path)
    texts = df['text'].tolist()
    labels = df['label'].tolist()
    return texts, labels

data, texts, labels = load_dataset(file_path)

benign_texts, _ = data[data['label'] == 0]["test"].tolist()
feature_extractor = EntropyKLFeatureExtractor(benign_texts)
features = np.array([feature_extractor.extract_features(text) for text in texts])

X_train, X_test, y_train, y_test = train_test_split(features,
                                                    labels,
                                                    test_size=0.3,
                                                    random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Training the model
model = EntropyClassifier(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

y_pred_labels = (np.array(all_preds) > 0.5).astype(int)
print(classification_report(all_labels, y_pred_labels))
print(f'ROC AUC Score: {roc_auc_score(all_labels, all_preds)}')